##  Spam

In [25]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import GridSearchCV

import joblib

In [26]:
url = "https://breathecode.herokuapp.com/asset/internal-link?id=932&path=url_spam.csv"

df = pd.read_csv(url)

df.head()

,url,is_spam
0,https://briefingday.us8.list-manage.com/unsubs...,True
1,https://www.hvper.com/,True
2,https://briefingday.com/m/v4n3i4f3,True
3,https://briefingday.com/n/20200618/m#commentform,False
4,https://briefingday.com/fan,True


# Exploracion de los datos

In [27]:
df.info()
df['is_spam'].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2999 entries, 0 to 2998
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   url      2999 non-null   object
 1   is_spam  2999 non-null   bool  
dtypes: bool(1), object(1)
memory usage: 26.5+ KB


is_spam
False    2303
True      696
Name: count, dtype: int64

El dataset muesta:
>
> - 2999 muestras
>
> - 2 columnas:

url (texto)

is_spam (booleano)

# Se convierte a enteros

In [28]:
df['is_spam'] = df['is_spam'].astype(int)

Convierto booleano en entero

In [29]:
df['is_spam'] = df['is_spam'].astype(int)

In [30]:
def preprocess_url(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['clean_url'] = df['url'].apply(preprocess_url)

df.head()

,url,is_spam,clean_url
0,https://briefingday.us8.list-manage.com/unsubs...,1,https briefingday us list manage com unsubs...
1,https://www.hvper.com/,1,https www hvper com
2,https://briefingday.com/m/v4n3i4f3,1,https briefingday com m v n i f
3,https://briefingday.com/n/20200618/m#commentform,0,https briefingday com n m commentform
4,https://briefingday.com/fan,1,https briefingday com fan


## Split

In [31]:
X = df['clean_url']
y = df['is_spam']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape


((2399,), (600,))

## Vectorización

In [32]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

X_train_vec.shape

(2399, 5000)

## Modelo SVM baseline

In [33]:
model = SVC(class_weight='balanced')

model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)

y_test, y_pred, y_test, y_pred

(509     0
 1358    0
 556     0
 790     0
 333     1
        ..
 2163    0
 2566    0
 494     1
 2609    0
 1168    0
 Name: is_spam, Length: 600, dtype: int64,
 array([0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0,
        0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0,
        0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0,
        0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
        0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1,
        0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1,
        0, 0, 1, 0, 0,

## Optimización con GridSearch

In [34]:
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf']
}

grid = GridSearchCV(
    SVC(class_weight='balanced'),
    param_grid,
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train_vec, y_train)

grid.best_params_

Fitting 3 folds for each of 6 candidates, totalling 18 fits


{'C': 10, 'kernel': 'rbf'}

## Mejor modelo

In [35]:
best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test_vec)

print("Optimized Accuracy:", accuracy_score(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))

Optimized Accuracy: 0.9466666666666667
              precision    recall  f1-score   support

           0       0.97      0.96      0.97       461
           1       0.87      0.91      0.89       139

    accuracy                           0.95       600
   macro avg       0.92      0.93      0.93       600
weighted avg       0.95      0.95      0.95       600



## Guardado

In [36]:
joblib.dump(best_model, "../models/url_spam_svm.pkl")

['../models/url_spam_svm.pkl']